# SafeCampus AI — Entrenamiento YOLOv8s (Gun + Knife)

Entrena un modelo **YOLOv8s** (small, 11.2M params) para detectar armas de fuego y cuchillos.

**Caracteristicas del notebook:**
- Resistente a desconexiones: cada celda es autonoma, resume automatico si Colab se corta
- Hiperparametros optimizados para weapon detection en CCTV
- AutoBatch: usa el maximo de VRAM disponible en T4 (15 GB)
- Checkpoints cada 5 epochs en Google Drive

**Pasos:**
1. Montar Google Drive + verificar GPU
2. Instalar dependencias
3. Descargar dataset de Roboflow
4. Entrenar YOLOv8s (con resume automatico)
5. Evaluar metricas
6. Probar con imagenes de test
7. Descargar `best.pt`

> **ANTES DE EJECUTAR:**
> 1. Ve a `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU T4`
> 2. Si vas a re-entrenar desde cero, borra la carpeta `SafeCampus-Training/gun_knife_v3/` en Google Drive
> 3. Para evitar desconexiones por inactividad, abre la consola del navegador (F12) y pega:
> ```javascript
> setInterval(() => { document.querySelector('colab-connect-button')?.click(); console.log('keep-alive'); }, 60000);
> ```

In [ ]:
# ==============================================================
# PASO 1: Montar Google Drive + verificar GPU
# ==============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, torch, gc

# Constantes globales — redefinidas en cada celda para ser autonomas
OUTPUT_DIR = "/content/drive/MyDrive/SafeCampus-Training"
RUN_NAME = "gun_knife_v3"
RESULTS_DIR = f"{OUTPUT_DIR}/{RUN_NAME}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Resultados se guardaran en: {RESULTS_DIR}")

# Limpiar VRAM por si hay basura de ejecuciones anteriores
torch.cuda.empty_cache()
gc.collect()

# Verificar GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")
else:
    print("ERROR: No hay GPU. Ve a Entorno de ejecucion > Cambiar tipo > GPU T4")

!nvidia-smi

In [ ]:
# ==============================================================
# PASO 2: Instalar dependencias (siempre usar ultima version)
# ==============================================================
# --upgrade es critico: versiones viejas de ultralytics tienen bugs
# con resume=True (NaN losses, KeyError 'step')
!pip install ultralytics roboflow --upgrade -q

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

In [ ]:
# ==============================================================
# PASO 3: Descargar dataset de Roboflow
# ==============================================================
#
# Dataset: "Gun and Knife Detection" por Mahad Ahmed
# Link: https://universe.roboflow.com/mahad-ahmed/gun-and-knife-detection
# ~8,451 imagenes | Clases: gun, knife
#
# Alternativas (descomenta si el principal no funciona):
# ALT 1 — Weapon yolo8 (EDI Detection) ~10,066 imgs
# project = rf.workspace("edi-detection").project("weapon-yolo8")
# ALT 2 — Weapon Detection (yolov7test) ~9,672 imgs
# project = rf.workspace("yolov7test-u13vc").project("weapon-detection-m7qso")

from roboflow import Roboflow

# Obtener tu API key gratis en: https://app.roboflow.com/settings/api
API_KEY = input("Ingresa tu Roboflow API key: ")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("mahad-ahmed").project("gun-and-knife-detection")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\nDataset descargado en: {dataset.location}")

In [ ]:
# ==============================================================
# PASO 3b: Verificar dataset + preparar data.yaml
# ==============================================================
import yaml
import os

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml, 'r') as f:
    config = yaml.safe_load(f)

print("=" * 40)
print("DATASET INFO")
print("=" * 40)
print(f"Clases ({config['nc']}): {config['names']}")

total = 0
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        total += count
        print(f"  {split}: {count} imagenes")
    else:
        print(f"  {split}: no encontrado")

print(f"  TOTAL: {total} imagenes")

# Guardar la ruta del data.yaml para el paso de evaluacion
# (si Colab se desconecta, el dataset sigue en /content/)
DATA_YAML_PATH = data_yaml
print(f"\ndata.yaml: {DATA_YAML_PATH}")

In [ ]:
# ==============================================================
# PASO 4: Entrenar YOLOv8s
# ==============================================================
#
# YOLOv8s (small): 11.2M params, buen balance precision/velocidad.
#
# RESUME AUTOMATICO: si Colab se desconecto y hay un last.pt guardado
# en Google Drive, esta celda lo detecta y continua desde ahi.
# IMPORTANTE: con resume=True NO se pasan otros argumentos; todos
# los hiperparametros se restauran del checkpoint.
#
# Si quieres EMPEZAR DE CERO:
#   1. Borra la carpeta SafeCampus-Training/gun_knife_v3/ en Google Drive
#   2. Reinicia el runtime (Ctrl+M .)
#   3. Ejecuta desde el Paso 1

from ultralytics import YOLO
import torch, gc, os

# Limpiar VRAM antes de entrenar
torch.cuda.empty_cache()
gc.collect()

OUTPUT_DIR = "/content/drive/MyDrive/SafeCampus-Training"
RUN_NAME = "gun_knife_v3"
LAST_PT = f"{OUTPUT_DIR}/{RUN_NAME}/weights/last.pt"

if os.path.exists(LAST_PT):
    # ---- RESUME: continuar entrenamiento interrumpido ----
    size_mb = os.path.getsize(LAST_PT) / 1024 / 1024
    print(f"Checkpoint encontrado: {LAST_PT} ({size_mb:.1f} MB)")
    print("Continuando entrenamiento desde el ultimo checkpoint...")
    print("(Todos los hiperparametros se restauran del checkpoint)\n")
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
else:
    # ---- ENTRENAMIENTO DESDE CERO ----
    print("Entrenamiento desde cero con yolov8s.pt")
    print(f"Resultados se guardaran en: {OUTPUT_DIR}/{RUN_NAME}/\n")
    model = YOLO("yolov8s.pt")
    results = model.train(
        data=data_yaml,
        epochs=150,
        imgsz=640,
        batch=-1,             # AutoBatch: usa el maximo de VRAM disponible (~60%)
        device=0,
        workers=2,            # Colab tiene CPU/RAM limitados para data loading
        patience=50,          # Early stopping: 50 epochs sin mejora (no 20, es muy agresivo)
        save=True,
        save_period=5,        # Checkpoint cada 5 epochs en Drive (seguro ante desconexiones)
        project=OUTPUT_DIR,
        name=RUN_NAME,
        exist_ok=True,        # Reusar carpeta (necesario para que resume funcione)
        # --- Optimizer ---
        optimizer="SGD",
        lr0=0.01,
        lrf=0.01,             # LR final = lr0 * lrf = 0.0001
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        cos_lr=True,          # Cosine annealing: LR decay suave, mejor convergencia
        nbs=64,               # Nominal batch size para normalizacion de loss
        # --- Augmentaciones optimizadas para weapon detection ---
        # Justificacion: armas son metalicas/oscuras, camaras CCTV son fijas
        hsv_h=0.015,          # Ligero shift de hue
        hsv_s=0.5,            # Saturacion moderada (armas tienen baja saturacion)
        hsv_v=0.3,            # Brillo moderado (armas son oscuras, no distorsionar)
        degrees=5.0,          # Rotacion baja (CCTV es angulo fijo)
        translate=0.1,        # Traslacion estandar
        scale=0.4,            # Escala moderada
        shear=0.0,            # Sin shear (CCTV fija)
        perspective=0.0,      # Sin perspectiva (CCTV fija)
        fliplr=0.5,           # Flip horizontal estandar
        flipud=0.0,           # Sin flip vertical (camaras no se invierten)
        mosaic=1.0,           # Mosaic activo: excelente para multi-escala
        mixup=0.1,            # MixUp bajo (mucho mixup confunde knife vs fondo)
        copy_paste=0.1,       # Copy-paste: pega armas sobre otros fondos
        close_mosaic=15,      # Desactiva mosaic 15 epochs antes del final
                              # para que el modelo ajuste bboxes precisos
    )

print("\n" + "=" * 50)
print("ENTRENAMIENTO COMPLETADO")
print("=" * 50)
print(f"Resultados en: {OUTPUT_DIR}/{RUN_NAME}/")
print(f"Modelo best: {OUTPUT_DIR}/{RUN_NAME}/weights/best.pt")

In [ ]:
# ==============================================================
# PASO 5: Ver resultados del entrenamiento
# ==============================================================
from IPython.display import Image, display
import os

OUTPUT_DIR = "/content/drive/MyDrive/SafeCampus-Training"
RUN_NAME = "gun_knife_v3"
RESULTS_DIR = f"{OUTPUT_DIR}/{RUN_NAME}"

plots = [
    ("Curvas de entrenamiento", "results.png", 800),
    ("Matriz de confusion", "confusion_matrix.png", 600),
    ("Matriz de confusion (normalizada)", "confusion_matrix_normalized.png", 600),
    ("Curva F1", "F1_curve.png", 600),
    ("Curva Precision-Recall", "PR_curve.png", 600),
    ("Predicciones en validacion", "val_batch0_pred.png", 800),
]

for title, filename, width in plots:
    path = f"{RESULTS_DIR}/{filename}"
    if os.path.exists(path):
        print(f"\n{title}:")
        display(Image(filename=path, width=width))
    else:
        print(f"\n{title}: no encontrado ({filename})")

In [ ]:
# ==============================================================
# PASO 6: Evaluar en test set
# ==============================================================
# Si Colab se desconecto y perdiste la variable data_yaml,
# necesitas re-descargar el dataset (Paso 3) o apuntar al yaml manualmente.

from ultralytics import YOLO
import os, glob

OUTPUT_DIR = "/content/drive/MyDrive/SafeCampus-Training"
RUN_NAME = "gun_knife_v3"
RESULTS_DIR = f"{OUTPUT_DIR}/{RUN_NAME}"
BEST_PT = f"{RESULTS_DIR}/weights/best.pt"

if not os.path.exists(BEST_PT):
    print(f"ERROR: No se encontro {BEST_PT}")
    print("Asegurate de que el entrenamiento (Paso 4) haya completado.")
else:
    best_model = YOLO(BEST_PT)
    print(f"Modelo cargado: {BEST_PT}")
    print(f"Clases: {best_model.names}")

    # Buscar data.yaml (puede estar en varias rutas segun la descarga)
    data_yaml_candidates = glob.glob("/content/**/data.yaml", recursive=True)
    if 'data_yaml' not in dir() or not os.path.exists(data_yaml):
        if data_yaml_candidates:
            data_yaml = data_yaml_candidates[0]
            print(f"data.yaml encontrado en: {data_yaml}")
        else:
            print("ERROR: No se encontro data.yaml. Re-ejecuta el Paso 3.")
            data_yaml = None

    if data_yaml and os.path.exists(data_yaml):
        metrics = best_model.val(data=data_yaml, device=0)

        print(f"\n{'=' * 50}")
        print(f"  RESULTADOS FINALES — SafeCampus YOLOv8s")
        print(f"{'=' * 50}")
        print(f"  mAP@50:      {metrics.box.map50:.3f}")
        print(f"  mAP@50-95:   {metrics.box.map:.3f}")
        print(f"  Precision:    {metrics.box.mp:.3f}")
        print(f"  Recall:       {metrics.box.mr:.3f}")
        print(f"{'=' * 50}")

        # Comparacion con modelo de referencia (Threat-Detection-YOLOv8n, HuggingFace)
        ref_map = 0.813
        ref_prec = 0.843
        ref_recall = 0.763
        new_map = metrics.box.map50
        new_prec = metrics.box.mp
        new_recall = metrics.box.mr

        print(f"\n  Comparacion con modelo anterior (HuggingFace YOLOv8n):")
        print(f"    mAP@50:    {ref_map:.3f} -> {new_map:.3f}  {'MEJOR' if new_map > ref_map else 'PEOR'}")
        print(f"    Precision: {ref_prec:.3f} -> {new_prec:.3f}  {'MEJOR' if new_prec > ref_prec else 'PEOR'}")
        print(f"    Recall:    {ref_recall:.3f} -> {new_recall:.3f}  {'MEJOR' if new_recall > ref_recall else 'PEOR'}")

        if new_map > ref_map and new_prec > ref_prec:
            print(f"\n  >>> MODELO NUEVO ES MEJOR. Usalo en SafeCampus. <<<")
        else:
            print(f"\n  >>> Modelo nuevo no supera al anterior en todas las metricas.")
            print(f"  >>> Considera: mas epochs, mas datos, o ajustar augmentaciones. <<<")

In [ ]:
# ==============================================================
# PASO 7: Probar con imagenes del test set
# ==============================================================
import glob, os
from ultralytics import YOLO
from IPython.display import display
from PIL import Image as PILImage

OUTPUT_DIR = "/content/drive/MyDrive/SafeCampus-Training"
RUN_NAME = "gun_knife_v3"
BEST_PT = f"{OUTPUT_DIR}/{RUN_NAME}/weights/best.pt"

if not os.path.exists(BEST_PT):
    print(f"ERROR: {BEST_PT} no encontrado")
else:
    best_model = YOLO(BEST_PT)

    # Buscar imagenes de test o validacion
    test_dirs = glob.glob("/content/**/test/images", recursive=True)
    val_dirs = glob.glob("/content/**/valid/images", recursive=True)

    img_dir = test_dirs[0] if test_dirs else (val_dirs[0] if val_dirs else None)

    if img_dir:
        images = glob.glob(os.path.join(img_dir, "*"))[:8]
        split_name = "test" if test_dirs else "validacion"
        print(f"Probando con {len(images)} imagenes de {split_name}:\n")

        results = best_model.predict(images, conf=0.50, device=0)
        for r in results:
            img = r.plot()
            display(PILImage.fromarray(img[:, :, ::-1]))
            print()
    else:
        print("No se encontraron imagenes. Re-ejecuta el Paso 3.")

In [ ]:
# ==============================================================
# PASO 8: Descargar best.pt
# ==============================================================
from google.colab import files
import shutil, os

OUTPUT_DIR = "/content/drive/MyDrive/SafeCampus-Training"
RUN_NAME = "gun_knife_v3"
BEST_PT = f"{OUTPUT_DIR}/{RUN_NAME}/weights/best.pt"
dst = "/content/best_safecampus_v3.pt"

if not os.path.exists(BEST_PT):
    print(f"ERROR: {BEST_PT} no encontrado")
else:
    shutil.copy2(BEST_PT, dst)

    size_mb = os.path.getsize(dst) / (1024 * 1024)
    print(f"Modelo: {dst}")
    print(f"Tamano: {size_mb:.1f} MB")
    print(f"\nTambien guardado en Google Drive: {BEST_PT}")
    print(f"\nInstrucciones:")
    print(f"  1. Descarga el archivo (se descarga automaticamente abajo)")
    print(f"  2. Copia a: safecampus-ai/backend/models/best.pt")
    print(f"  3. Reinicia Flask para que cargue el modelo nuevo")

    files.download(dst)

---

## Celda de emergencia: Resume rapido

Si Colab se desconecto y perdiste todas las variables, ejecuta **solo esta celda**.
Monta Drive, instala dependencias y retoma el entrenamiento desde el ultimo checkpoint.

In [ ]:
# ==============================================================
# CELDA DE EMERGENCIA: Resume tras desconexion de Colab
# ==============================================================
# Ejecuta SOLO esta celda si Colab se desconecto a mitad del
# entrenamiento. Hace todo lo necesario para retomar.

from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics --upgrade -q

import os, torch, gc
torch.cuda.empty_cache()
gc.collect()

from ultralytics import YOLO

OUTPUT_DIR = "/content/drive/MyDrive/SafeCampus-Training"
RUN_NAME = "gun_knife_v3"
LAST_PT = f"{OUTPUT_DIR}/{RUN_NAME}/weights/last.pt"

if os.path.exists(LAST_PT):
    size_mb = os.path.getsize(LAST_PT) / 1024 / 1024
    print(f"Checkpoint: {LAST_PT} ({size_mb:.1f} MB)")
    print("Retomando entrenamiento...\n")
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
    print("\nEntrenamiento completado!")
else:
    print(f"ERROR: No hay checkpoint en {LAST_PT}")
    print("El entrenamiento no habia empezado o la carpeta fue borrada.")
    print("Ejecuta el notebook completo desde el Paso 1.")